# Netflix Prize — LightFM (WARP) ranking experiment

MF and kNN optimize **rating accuracy** (RMSE) and rank as a side effect. **LightFM with
WARP loss optimizes ranking directly** — so the question is whether it beats MF on
**sampled-neg MAP@10** (MF got ~0.31). It does *not* predict a star rating, so there's no
RMSE here and it can't join the rating-blend directly (it could join a *rank* blend).

Reuses the same parse / split / metrics as the other notebooks (same `SEED`).

> Not verified locally (LightFM needs a Cython build absent on the dev machine) — run on Colab.

In [ ]:
!pip install -q lightfm
# If the build fails on a very new runtime, try:  !pip install -q lightfm --no-build-isolation

## 0. Setup + parse + split (shared)

In [ ]:
import os, time
import numpy as np
import pandas as pd
from scipy.sparse import coo_matrix

DATA_DIR  = '/content/netflix'      # <-- EDIT ME
CACHE_NPZ = os.path.join(DATA_DIR, 'ratings_cache.npz')
CHUNK_ROWS = 5_000_000
SUBSAMPLE_FILES = [1, 2, 3, 4]
USER_FRAC = 1.0
SEED = 42
TEST_FRAC = 0.20
RELEVANT_THRESHOLD = 3.5

# LightFM hyperparameters
N_COMPONENTS = 100
EPOCHS_LFM   = 20
POS_THRESHOLD = 4.0     # rating >= this counts as a positive interaction (WARP is implicit)
NUM_THREADS  = 4
rng = np.random.default_rng(SEED)
print('config loaded')

In [ ]:
def parse_combined(path, chunksize=CHUNK_ROWS):
    us, ms, rs, ds = [], [], [], []; cur = 0
    for chunk in pd.read_csv(path, header=None, names=['user','rating','date'],
                             dtype={'user': str}, chunksize=chunksize):
        is_hdr = chunk['rating'].isna()
        mv = pd.to_numeric(chunk['user'].where(is_hdr).str.rstrip(':'),
                           errors='coerce').ffill().fillna(cur)
        r = ~is_hdr
        us.append(pd.to_numeric(chunk.loc[r,'user']).to_numpy(np.int32))
        ms.append(mv[r].to_numpy(np.int16)); rs.append(chunk.loc[r,'rating'].to_numpy(np.int8))
        ds.append(chunk.loc[r,'date'].to_numpy('datetime64[D]')); cur = int(mv.iloc[-1])
    return (np.concatenate(us), np.concatenate(ms), np.concatenate(rs), np.concatenate(ds))

if os.path.exists(CACHE_NPZ):
    z = np.load(CACHE_NPZ)
    raw_user, movie, rating, date = z['user'], z['movie'], z['rating'], z['date']
else:
    pu,pm,pr,pd_ = [],[],[],[]
    for n in SUBSAMPLE_FILES:
        u,m,r,d = parse_combined(os.path.join(DATA_DIR, f'combined_data_{n}.txt'))
        pu.append(u); pm.append(m); pr.append(r); pd_.append(d)
    raw_user=np.concatenate(pu); movie=np.concatenate(pm)
    rating=np.concatenate(pr); date=np.concatenate(pd_)
    np.savez(CACHE_NPZ, user=raw_user, movie=movie, rating=rating, date=date)

_, user = np.unique(raw_user, return_inverse=True); user = user.astype(np.int32); del raw_user
movie = (movie - 1).astype(np.int32)
if USER_FRAC < 1.0:
    keep = np.zeros(user.max()+1, bool)
    keep[rng.choice(keep.size, int(keep.size*USER_FRAC), replace=False)] = True
    mr = keep[user]; user, movie, rating, date = user[mr], movie[mr], rating[mr], date[mr]
    _, user = np.unique(user, return_inverse=True); user = user.astype(np.int32)
n_users = int(user.max())+1; n_movies = int(movie.max())+1
print(f'{len(rating):,} ratings | {n_users:,} users | {n_movies:,} movies')

order = np.lexsort((date, user)); su = user[order]
_, st, ct = np.unique(su, return_index=True, return_counts=True)
st = st.astype(np.int32); ct = ct.astype(np.int32)
pos = np.arange(len(order), dtype=np.int32) - np.repeat(st, ct)
nt = np.where(ct >= 5, np.ceil(ct*TEST_FRAC).astype(np.int32), 0)
is_test = np.zeros(len(order), bool); is_test[order] = pos >= np.repeat(ct-nt, ct)
del order, su, pos
trm = ~is_test
u_tr, m_tr, r_tr = user[trm], movie[trm], rating[trm]
u_te, m_te, r_te = user[is_test], movie[is_test], rating[is_test]
print(f'train {trm.sum():,} | test {is_test.sum():,}')

## 1. MAP@10 metrics (same as the MF/kNN notebooks)

In [ ]:
def map_at_k(u_te, m_te, r_te, predict_fn, k=10, max_users=20000, seed=SEED):
    pred = predict_fn(u_te, m_te)
    o = np.argsort(u_te, kind='stable'); uu, rr, pp = u_te[o], r_te[o], pred[o]
    users_u, st = np.unique(uu, return_index=True); en = np.append(st[1:], len(uu))
    rgn = np.random.default_rng(seed); sel = np.arange(len(users_u))
    if len(sel) > max_users: sel = rgn.choice(sel, max_users, replace=False)
    aps = []
    for j in sel:
        s, e = st[j], en[j]
        if e - s < 2: continue
        rel = rr[s:e] >= RELEVANT_THRESHOLD; R = int(rel.sum())
        if R == 0: continue
        rank = np.argsort(-pp[s:e], kind='stable')[:k]
        hits = 0; ap = 0.0
        for i, ri in enumerate(rank, 1):
            if rel[ri]: hits += 1; ap += hits / i
        aps.append(ap / min(k, R))
    return float(np.mean(aps)), len(aps)

def map_at_k_sampled(predict_fn, u_tr, m_tr, u_te, m_te, r_te,
                     k=10, n_neg=100, max_users=10000, seed=SEED):
    rgn = np.random.default_rng(seed)
    o = np.argsort(u_tr, kind='stable'); su, sm = u_tr[o], m_tr[o]
    bounds = np.searchsorted(su, np.arange(n_users + 1))
    ot = np.argsort(u_te, kind='stable'); tu, tm, trr = u_te[ot], m_te[ot], r_te[ot]
    tusers, tst = np.unique(tu, return_index=True); ten = np.append(tst[1:], len(tu))
    sel = np.arange(len(tusers))
    if len(sel) > max_users: sel = rgn.choice(sel, max_users, replace=False)
    aps = []
    for j in sel:
        uidx = int(tusers[j]); s, e = tst[j], ten[j]
        t_movies, t_r = tm[s:e], trr[s:e]
        rel = t_r >= RELEVANT_THRESHOLD; R = int(rel.sum())
        if R == 0: continue
        seen = set(sm[bounds[uidx]:bounds[uidx + 1]].tolist()) | set(t_movies.tolist())
        negs = []
        while len(negs) < n_neg:
            for c in rgn.integers(0, n_movies, n_neg):
                c = int(c)
                if c not in seen:
                    negs.append(c); seen.add(c)
                    if len(negs) >= n_neg: break
        cand = np.concatenate([t_movies, np.array(negs, dtype=np.int32)])
        labels = np.concatenate([rel, np.zeros(len(negs), bool)])
        scores = predict_fn(np.full(len(cand), uidx, np.int32), cand)
        rank = np.argsort(-scores, kind='stable')[:k]
        hits = 0; ap = 0.0
        for i, ri in enumerate(rank, 1):
            if labels[ri]: hits += 1; ap += hits / i
        aps.append(ap / min(k, R))
    return float(np.mean(aps)), len(aps)

## 2. Train LightFM (WARP)

WARP needs **implicit positives**: we treat `rating >= POS_THRESHOLD` as a positive
interaction (a strong 'like'). WARP samples negatives and pushes positives above them —
directly optimizing the top of the ranking, which is what MAP@10 rewards.

In [ ]:
from lightfm import LightFM

pos = r_tr >= POS_THRESHOLD
interactions = coo_matrix((np.ones(int(pos.sum()), np.float32),
                           (u_tr[pos], m_tr[pos])), shape=(n_users, n_movies))
print(f'{interactions.nnz:,} positive interactions (rating >= {POS_THRESHOLD})')

lfm = LightFM(loss='warp', no_components=N_COMPONENTS, random_state=SEED)
t0 = time.time()
lfm.fit(interactions, epochs=EPOCHS_LFM, num_threads=NUM_THREADS, verbose=True)
print(f'trained in {time.time()-t0:.0f}s')

## 3. Evaluate ranking (sampled-neg MAP@10) vs baseline

In [ ]:
def predict_lightfm(u_arr, m_arr):
    return lfm.predict(np.asarray(u_arr, np.int32), np.asarray(m_arr, np.int32),
                       num_threads=NUM_THREADS)

# popularity baseline for a reference floor (item frequency = pure popularity ranking)
pop = np.bincount(m_tr, minlength=n_movies).astype(np.float32)
def predict_pop(u_arr, m_arr):
    return pop[np.asarray(m_arr)]

s_ho, n_ho = map_at_k(u_te, m_te, r_te, predict_lightfm)
s_sn, n_sn = map_at_k_sampled(predict_lightfm, u_tr, m_tr, u_te, m_te, r_te)
p_sn, _    = map_at_k_sampled(predict_pop,      u_tr, m_tr, u_te, m_te, r_te)
print(f'LightFM(WARP) MAP@10  held-out={s_ho:.4f} ({n_ho})  sampled-neg={s_sn:.4f} ({n_sn})')
print(f'popularity     MAP@10  sampled-neg={p_sn:.4f}')
print(f'(MF from the main notebook: sampled-neg ~0.31 — compare here)')

## 4. Notes

- **If LightFM's sampled-neg MAP@10 > MF's ~0.31**, ranking-aware training helped — report
  it as the strongest *ranking* model even though MF wins on RMSE. That RMSE-vs-MAP split is
  exactly the rating-accuracy-vs-ranking trade-off the rubric asks you to discuss.
- **Incorporating it**: LightFM scores aren't ratings, so it can't go into the RMSE ridge
  blend. Two ways to use it: (a) serve MF for predicted ratings + LightFM for the Top-K list;
  (b) a *rank-level* blend (average the per-user item ranks from MF and LightFM).
- **Hybrid**: LightFM's real edge is item/user **features** — feed movie year/decade (and
  later your IMDB genre/language join) via `item_features` to fight cold-start. That's the
  natural bridge to the hybrid-recommender optional task.
- Tune `POS_THRESHOLD` (3.5 vs 4.0), `no_components`, `epochs`; try `loss='bpr'` for contrast.